# Chapter 15 — Building a Multi-Step Essay-Writer Agent

Hands-On LangChain. Code targets **LangGraph 1.x**.

In the final LangGraph chapter we put everything together into a more complex system: an **essay writer**
made of several specialized roles wired into one graph. It **plans** an outline, **researches** the
topic, **writes** a draft, **reflects** on it like a critic, **researches again** based on that critique,
and **revises** — looping until it has produced a polished essay. This brings together planning, tool
use, reflection, and cyclical graphs.

Companion notebook for Chapter 15.

> Modernizations from the source: research uses **DuckDuckGo** (no key) instead of Tavily; the `Queries`
> schema imports from **`pydantic`** (the old `langchain_core.pydantic_v1` shim is gone in 1.x);
> persistence uses **`InMemorySaver`**.

## 15.1 Project setup and dependencies

Configure Ollama and pull the chat model set in `MODEL` in `.env` before running this notebook.

In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from model_provider import get_chat_model

memory = InMemorySaver()
model = get_chat_model(temperature=0)

## 15.2 Defining the agent's state

The state is the backbone of the agent — the data passed between nodes. Our essay writer tracks the
task, the plan, gathered research `content`, the current `draft`, the latest `critique`, and a revision
counter with a maximum.

In [2]:
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

## 15.3 Creating the agent roles (prompts and tools)

Each role is driven by a focused system prompt.

In [3]:
PLAN_PROMPT = (
    "You are an expert writer tasked with writing a high level outline of an essay. "
    "Write such an outline for the user provided topic. Give an outline of the essay along "
    "with any relevant notes or instructions for the sections."
)

WRITER_PROMPT = (
    "You are an essay assistant tasked with writing excellent 5-paragraph essays. "
    "Generate the best essay possible for the user's request and the initial outline. "
    "If the user provides critique, respond with a revised version of your previous attempts. "
    "Utilize all the information below as needed:\n------\n{content}"
)

REFLECTION_PROMPT = (
    "You are a teacher grading an essay submission. Generate critique and recommendations for "
    "the user's submission. Provide detailed recommendations, including requests for length, "
    "depth, style, etc."
)

RESEARCH_PLAN_PROMPT = (
    "You are a researcher charged with providing information that can be used when writing the "
    "following essay. Generate a list of search queries that will gather any relevant "
    "information. Only generate 3 queries max."
)

RESEARCH_CRITIQUE_PROMPT = (
    "You are a researcher charged with providing information that can be used when making any "
    "requested revisions (as outlined below). Generate a list of search queries that will gather "
    "relevant information. Only generate 3 queries max."
)

The research nodes generate a list of search queries, so we define a Pydantic model for that structured output, and a small DuckDuckGo helper to run the searches.

In [4]:
from pydantic import BaseModel
from ddgs import DDGS

class Queries(BaseModel):
    queries: List[str]

def web_search(query, max_results=2):
    """Return a few result snippets for a query (empty list if the search fails)."""
    try:
        with DDGS() as ddgs:
            return [r["body"] for r in ddgs.text(query, max_results=max_results)]
    except Exception as e:
        print(f"  (search failed for {query!r}: {e})")
        return []

## 15.4 Implementing the node functions

Each function takes the current `AgentState` and returns a dict of updated state values.

In [5]:
def plan_node(state: AgentState):
    messages = [SystemMessage(content=PLAN_PROMPT), HumanMessage(content=state["task"])]
    response = model.invoke(messages)
    return {"plan": response.content}

def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state["task"]),
    ])
    content = state.get("content") or []
    for q in queries.queries:
        content.extend(web_search(q))
    return {"content": content}

def generation_node(state: AgentState):
    content = "\n\n".join(state.get("content") or [])
    user_message = HumanMessage(content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [SystemMessage(content=WRITER_PROMPT.format(content=content)), user_message]
    response = model.invoke(messages)
    return {"draft": response.content, "revision_number": state.get("revision_number", 1) + 1}

def reflection_node(state: AgentState):
    messages = [SystemMessage(content=REFLECTION_PROMPT), HumanMessage(content=state["draft"])]
    response = model.invoke(messages)
    return {"critique": response.content}

def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state["critique"]),
    ])
    content = state.get("content") or []
    for q in queries.queries:
        content.extend(web_search(q))
    return {"content": content}

## 15.5 Defining the graph's logic and edges

The essay writer needs a **revision loop**. A `should_continue` function decides whether to revise again
or finish, based on the revision count. Then we wire the nodes into a graph: plan → research → generate,
and from generate either end or go reflect → research the critique → generate again.

In [6]:
def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

builder = StateGraph(AgentState)
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_conditional_edges("generate", should_continue, {END: END, "reflect": "reflect"})
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

graph = builder.compile(checkpointer=memory)

## 15.6 Visualizing and running the agent

First, the graph structure (as a Mermaid diagram) — note the revision loop from `generate` through
`reflect` and `research_critique` and back.

In [7]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	generate(generate)
	reflect(reflect)
	research_plan(research_plan)
	research_critique(research_critique)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	generate -.-> __end__;
	generate -.-> reflect;
	planner --> research_plan;
	reflect --> research_critique;
	research_critique --> generate;
	research_plan --> generate;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Now run it. We ask for an essay on the difference between AI and Generative AI, allowing one
revision (`max_revisions: 2`). We print a short summary of each node as it completes.

In [ ]:
thread = {"configurable": {"thread_id": "1"}}
inputs = {
    "task": "What is the difference between AI and Generative AI?",
    "max_revisions": 2,
    "revision_number": 1,
}

for step in graph.stream(inputs, thread):
    for node, update in step.items():
        if node == "planner":
            print("PLANNER — outline:\n", update["plan"][:400], "...\n")
        elif node in ("research_plan", "research_critique"):
            print(f"{node.upper()} — gathered {len(update['content'])} research snippets total\n")
        elif node == "generate":
            print(f"GENERATE — draft (revision {update['revision_number'] - 1}):\n",
                  update["draft"][:400], "...\n")
        elif node == "reflect":
            print("REFLECT — critique:\n", update["critique"][:400], "...\n")

Finally, we can pull the finished essay from the graph's saved state:

In [ ]:
final = graph.get_state(thread).values["draft"]
print(final[:1200])

The agent produced a researched, revised essay by routing through several specialized roles in a
loop — planning, researching, writing, self-critiquing, and revising — all coordinated by the graph's
state and edges.

This completes our LangGraph journey: from a single ReAct agent to a stateful, multi-step system that
plans, uses tools, reflects, and revises. These same building blocks scale up to the sophisticated
agentic applications you'll build in the real world.